In [ ]:
%pip install -q \
  extremeweatherbench \
  icechunk \
  xarray \
  pandas \
  numpy \
  scipy \
  scikit-learn \
  matplotlib

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

print("===== 라이브러리 확인 =====")
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Xarray:", xr.__version__)

# NumPy 기본 계산 테스트
a = np.array([1, 2, 3])
print("NumPy 계산:", np.mean(a))

# Xarray 생성 테스트
test = xr.DataArray(
    np.array([10, 20, 30]),
    dims=["time"]
)

print("Xarray 테스트:", test.values)

print("\n기본 라이브러리 정상 작동!")

In [ ]:
import numba
import numpy
import pandas
import xarray
import icechunk
import extremeweatherbench

from extremeweatherbench import inputs
from sklearn.metrics import mean_absolute_error

print("Numba:", numba.__version__)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("Xarray:", xarray.__version__)

print("ExtremeWeatherBench 불러오기 성공")
print("Icechunk 불러오기 성공")

print(
    "CIRA 예측자료 연결 기능:",
    hasattr(inputs, "get_cira_icechunk")
)

In [ ]:
import icechunk
import extremeweatherbench

from extremeweatherbench import inputs

print("Icechunk 정상")
print("ExtremeWeatherBench 정상")

print(
    "CIRA 데이터 연결 기능:",
    hasattr(inputs, "get_cira_icechunk")
)

In [ ]:
from google.colab import drive
from pathlib import Path

import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/weather_ai")

AWS_DIR = ROOT / "aws_csv"
WORK_DIR = ROOT / "evaluation_3models"

PRED_DIR = WORK_DIR / "predictions"
TABLE_DIR = WORK_DIR / "tables"
FIG_DIR = WORK_DIR / "figures"

for folder in [PRED_DIR, TABLE_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

MODELS = {
    "GraphCast": "GRAP_v100_GFS",
    "Pangu-Weather": "PANG_v100_GFS",
    "FourCastNet": "FOUR_v200_GFS",
}

YEAR = 2023
LEAD_HOURS = 24

# 첫 실행은 공통 날짜 3일만 확인
SAMPLE_DAYS = None

print("평가 모델:", list(MODELS))
print("평가연도:", YEAR)
print("예측시간:", LEAD_HOURS, "시간")

In [ ]:
from pathlib import Path

AWS_DIR = ROOT / "aws_csv"

print("===== AWS CSV 파일 목록 =====")

all_csv_files = sorted(AWS_DIR.rglob("*.csv"))

for path in all_csv_files:
    if "2023" in path.name or "2024" in path.name:
        print(path.name)

print("\n전체 CSV 파일 수:", len(all_csv_files))

In [ ]:
# ==========================================
# 셀 3. 기상청 AWS 실제 관측자료 불러오기
# ==========================================

import pandas as pd
import numpy as np
from pathlib import Path

YEAR = 2023
AWS_DIR = ROOT / "aws_csv"

# 1. CSV 파일 전체 검색
all_csv_files = sorted(AWS_DIR.rglob("*.csv"))

# 2. 2023년 파일 검색
files_2023 = [
    p for p in all_csv_files
    if str(YEAR) in p.name
]

# 3. 2024년 파일 검색
# 2023-12-31 예측의 정답은 2024-01-01
files_2024 = [
    p for p in all_csv_files
    if str(YEAR + 1) in p.name
]

print("===== AWS 데이터 파일 확인 =====")

print("\n2023년 파일:", len(files_2023))
for p in files_2023:
    print("-", p.name)

print("\n2024년 파일:", len(files_2024))
for p in files_2024:
    print("-", p.name)

# 2023년 파일이 없으면 즉시 중단
if not files_2023:
    raise FileNotFoundError(
        "2023년 AWS CSV 파일을 찾지 못했습니다. "
        "weather_ai/aws_csv 폴더를 확인하세요."
    )

csv_files = files_2023 + files_2024

# 4. CSV에서 오전 9시 관측자료 추출
parts = []

for path in csv_files:

    print("\n읽는 중:", path.name)

    file_count = 0

    for chunk in pd.read_csv(
        path,
        encoding="cp949",
        usecols=[
            "지점",
            "지점명",
            "일시",
            "기온(°C)"
        ],
        chunksize=100_000
    ):

        # 관측시간 변환
        times = pd.to_datetime(
            chunk["일시"],
            errors="coerce"
        )

        # 평가기간:
        # 2023년 전체 + 2024년 1월 1일
        date_mask = (
            (times.dt.year == YEAR)
            |
            (
                (times.dt.year == YEAR + 1)
                & (times.dt.month == 1)
                & (times.dt.day == 1)
            )
        )

        # UTC 00시 초기화 + 24시간 예측
        # 실제 시각: 다음날 KST 09시
        time_mask = (
            (times.dt.hour == 9)
            & (times.dt.minute == 0)
        )

        mask = date_mask & time_mask

        if not mask.any():
            continue

        selected = chunk.loc[mask].copy()

        selected["valid_time_kst"] = (
            times.loc[mask]
            .dt.tz_localize("Asia/Seoul")
        )

        selected["station_id"] = pd.to_numeric(
            selected["지점"],
            errors="coerce"
        )

        selected["actual_c"] = pd.to_numeric(
            selected["기온(°C)"],
            errors="coerce"
        )

        selected["station_name"] = (
            selected["지점명"]
            .astype(str)
            .str.strip()
        )

        selected = selected[
            [
                "station_id",
                "station_name",
                "valid_time_kst",
                "actual_c"
            ]
        ]

        parts.append(selected)

        file_count += len(selected)

    print("추출 건수:", file_count)


# 5. 데이터 통합
if not parts:
    raise ValueError(
        "2023년 AWS 오전 9시 관측자료가 없습니다. "
        "CSV 내부 날짜 형식을 확인하세요."
    )

aws = pd.concat(
    parts,
    ignore_index=True
)

# 6. 결측값 제거
aws = aws.dropna(
    subset=[
        "station_id",
        "valid_time_kst",
        "actual_c"
    ]
).copy()

aws["station_id"] = (
    aws["station_id"].astype(int)
)

# 비정상 기온 검사
aws = aws[
    aws["actual_c"].between(-50, 50)
].copy()

# 7. 중복 검사
keys = [
    "station_id",
    "valid_time_kst"
]

conflicts = (
    aws.groupby(keys)["actual_c"]
    .nunique()
)

if (conflicts > 1).any():
    raise ValueError(
        "동일 관측소·시각에 서로 다른 관측 기온이 존재합니다."
    )

aws = aws.drop_duplicates(
    subset=keys
)

aws = aws.sort_values(
    ["valid_time_kst", "station_id"]
).reset_index(drop=True)

# 8. 2023년 자료 확보 여부 확인
aws_2023 = aws[
    aws["valid_time_kst"].dt.year == YEAR
]

if aws_2023.empty:
    raise ValueError(
        "파일은 발견했지만 2023년 관측자료가 없습니다. "
        "CSV의 '일시' 컬럼을 확인하세요."
    )

print("\n===== AWS 데이터 확인 =====")

print("전체 관측 건수:", len(aws))
print("2023년 관측 건수:", len(aws_2023))

print(
    "관측 날짜 수:",
    aws_2023["valid_time_kst"]
    .dt.date.nunique()
)

print(
    "관측소 수:",
    aws_2023["station_id"].nunique()
)

print(
    "최초 관측:",
    aws_2023["valid_time_kst"].min()
)

print(
    "마지막 관측:",
    aws_2023["valid_time_kst"].max()
)

display(aws.head(10))

In [ ]:
# AI 예측의 24시간 뒤 실제 시각
forecast_times = (
    pd.to_datetime(
        predictions["forecast_init_utc"],
        utc=True
    )
    + pd.Timedelta(hours=24)
).dt.tz_convert("Asia/Seoul")

pred_dates = set(
    forecast_times.dt.strftime("%Y-%m-%d")
)

aws_dates = set(
    aws["valid_time_kst"].dt.strftime("%Y-%m-%d")
)

common_dates = sorted(
    pred_dates & aws_dates
)

print("===== 날짜 비교 =====")

print("AI 예측 날짜:", sorted(pred_dates))
print("AWS 관측 날짜 일부:", sorted(aws_dates)[:5])
print("공통 날짜:", common_dates)

if not common_dates:
    raise ValueError(
        "아직 공통 날짜가 없습니다. "
        "AWS 파일의 실제 관측기간을 확인하세요."
    )

print("\n날짜 일치 확인 완료!")

In [ ]:
station_path = ROOT / "station_master_review.csv"

if not station_path.exists():

    template = (
        aws[["station_id", "station_name"]]
        .drop_duplicates("station_id")
        .sort_values("station_id")
    )

    template["latitude"] = np.nan
    template["longitude"] = np.nan

    template.to_csv(
        station_path,
        index=False,
        encoding="utf-8-sig"
    )

    raise RuntimeError(
        "관측소 정보 검토용 파일을 생성했습니다. "
        "기상청 공식 위도·경도를 확인해 입력한 후 "
        "셀 4를 다시 실행하세요."
    )

stations = pd.read_csv(station_path)

required = [
    "station_id", "latitude", "longitude"
]

if not set(required).issubset(stations.columns):
    raise ValueError("관측소 정보 파일의 컬럼을 확인하세요.")

stations = stations[required].copy()

stations = stations.dropna(
    subset=["latitude", "longitude"]
).copy()

stations["station_id"] = stations["station_id"].astype(int)

if stations["station_id"].duplicated().any():
    raise ValueError("중복 관측소 번호가 있습니다.")

if not (
    stations["latitude"].between(33, 39).all()
    and stations["longitude"].between(124, 132).all()
):
    raise ValueError("관측소 위도·경도 범위를 확인하세요.")

stations = stations[
    stations["station_id"].isin(aws["station_id"])
].sort_values("station_id").reset_index(drop=True)

if stations.empty:
    raise ValueError("평가 가능한 관측소가 없습니다.")

print("선정 관측소:", len(stations))
display(stations)

In [ ]:
from extremeweatherbench import inputs

forecast_datasets = {}

for model_name, group_name in MODELS.items():

    print(f"\n===== {model_name} 연결 =====")

    # NOAA/CIRA 가상 예측자료 연결
    forecast = inputs.get_cira_icechunk(
        model_name=group_name
    )

    # 실제 데이터는 필요한 시점에 읽도록 구성
    ds = forecast.open_and_maybe_preprocess_data_from_source()

    print("차원:", dict(ds.sizes))
    print("변수:", list(ds.data_vars))

    forecast_datasets[model_name] = ds

print("\n3개 모델 데이터 연결 완료")

In [ ]:
TEMP_NAMES = [
    "surface_air_temperature",
    "t2m",
    "t2"
]

MODEL_TEMP_COL = {}

for name, ds in forecast_datasets.items():

    print(f"\n===== {name} =====")

    temp_col = next(
        (v for v in TEMP_NAMES if v in ds.data_vars),
        None
    )

    if temp_col is None:
        raise ValueError(
            f"{name}: 2m 기온 변수를 찾을 수 없습니다. "
            f"실제 변수: {list(ds.data_vars)}"
        )

    for coord in [
        "init_time",
        "lead_time",
        "latitude",
        "longitude"
    ]:
        if coord not in ds.coords:
            raise ValueError(
                f"{name}: {coord} 좌표가 없습니다. "
                f"실제 좌표: {list(ds.coords)}"
            )

    MODEL_TEMP_COL[name] = temp_col

    print("기온 변수:", temp_col)
    print("기온 단위:", ds[temp_col].attrs.get("units"))
    print("예측시간 일부:", ds["lead_time"].values[:8])

print("\n세 모델 구조 확인 완료")

In [ ]:
date_sets = []

for name, ds in forecast_datasets.items():

    times = pd.DatetimeIndex(
        ds["init_time"].values
    )

    times = times[
        (times.year == YEAR)
        & (times.hour == 0)
    ]

    date_sets.append(set(times.to_numpy()))

    print(
        name,
        "해당 연도 초기 예측시각:",
        len(times)
    )

# 세 모델 모두 보유한 시각만
common_dates = sorted(
    set.intersection(*date_sets)
)

if not common_dates:
    raise ValueError(
        "세 모델의 공통 예측 날짜가 없습니다."
    )

# 처음에는 3일만 테스트
selected_dates = (
    common_dates
    if SAMPLE_DAYS is None
    else common_dates[:SAMPLE_DAYS]
)

selected_dates = np.asarray(
    selected_dates,
    dtype="datetime64[ns]"
)

print("\n전체 공통 날짜:", len(common_dates))
print("이번 실행 날짜:", len(selected_dates))
print("예측 시작일:", selected_dates[0])
print("예측 종료일:", selected_dates[-1])

In [ ]:
def extract_station_temperature(
    ds,
    temp_col,
    dates,
    model_name
):

    temp = ds[temp_col]

    # 24시간 뒤 예측만 선택
    leads = temp["lead_time"].values

    if not np.issubdtype(leads.dtype, np.timedelta64):
        raise ValueError(
            f"{model_name}: 예측시간 단위가 timedelta이 아닙니다."
        )

    target_lead = np.timedelta64(
        LEAD_HOURS, "h"
    )

    if not np.any(leads == target_lead):
        raise ValueError(
            f"{model_name}: 24시간 예측이 없습니다."
        )

    temp = temp.sel(
        lead_time=target_lead,
        init_time=dates
    )

    # 서울 주변만 선택
    lat_min = stations.latitude.min() - 0.5
    lat_max = stations.latitude.max() + 0.5
    lon_min = stations.longitude.min() - 0.5
    lon_max = stations.longitude.max() + 0.5

    latitude = temp["latitude"].values
    longitude = temp["longitude"].values

    lat_slice = (
        slice(lat_min, lat_max)
        if latitude[0] < latitude[-1]
        else slice(lat_max, lat_min)
    )

    lon_slice = (
        slice(lon_min, lon_max)
        if longitude[0] < longitude[-1]
        else slice(lon_max, lon_min)
    )

    temp = temp.sel(
        latitude=lat_slice,
        longitude=lon_slice
    )

    # 관측소 좌표로 이중선형 보간
    temp = temp.interp(
        latitude=xr.DataArray(
            stations.latitude.to_numpy(),
            dims="station"
        ),
        longitude=xr.DataArray(
            stations.longitude.to_numpy(),
            dims="station"
        ),
        method="linear"
    )

    # 필요한 시점과 지점의 데이터만 가져옴
    values = temp.transpose(
        "init_time", "station"
    ).compute().values.astype(float)

    if values.shape != (
        len(dates), len(stations)
    ):
        raise ValueError(
            f"{model_name}: 예측 결과 배열 크기 불일치"
        )

    if not np.isfinite(values).all():
        raise ValueError(
            f"{model_name}: 유효하지 않은 기온값이 있습니다."
        )

    # 기온 단위 변환
    units = str(
        temp.attrs.get("units", "")
    ).strip().lower()

    if units in ["k", "kelvin"]:
        values = values - 273.15

    elif units in [
        "c", "degc", "degree_celsius",
        "degrees_celsius", "°c"
    ]:
        pass

    else:
        raise ValueError(
            f"{model_name}: 기온 단위 확인 필요: {units!r}"
        )

    if not np.all(
        (values > -100) & (values < 70)
    ):
        raise ValueError(
            f"{model_name}: 기온 값 범위 확인 필요"
        )

    return pd.DataFrame({
        "model": model_name,
        "forecast_init_utc": np.repeat(
            dates, len(stations)
        ),
        "station_id": np.tile(
            stations.station_id.to_numpy(),
            len(dates)
        ),
        "prediction_c": values.reshape(-1)
    })

In [ ]:
all_parts = []

for model_name, ds in forecast_datasets.items():

    print(f"\n===== {model_name} 예측값 추출 =====")

    result = extract_station_temperature(
        ds=ds,
        temp_col=MODEL_TEMP_COL[model_name],
        dates=selected_dates,
        model_name=model_name
    )

    path = (
        PRED_DIR /
        f"{model_name.replace('-', '_')}_predictions.csv"
    )

    result.to_csv(
        path,
        index=False,
        encoding="utf-8-sig"
    )

    all_parts.append(result)

    print("저장:", path.name)
    print("예측 건수:", len(result))


predictions = pd.concat(
    all_parts,
    ignore_index=True
)

predictions["forecast_init_utc"] = pd.to_datetime(
    predictions["forecast_init_utc"],
    utc=True
)

print("\n===== 세 모델 확보 결과 =====")

display(
    predictions.groupby("model").agg(
        예측건수=("station_id", "size"),
        날짜수=("forecast_init_utc", "nunique"),
        관측소수=("station_id", "nunique")
    )
)

#14번

In [ ]:
import pandas as pd

# 예측 날짜와 시간 확인
print("===== AI 예측자료 =====")
print("예측 시작시각:")
print(predictions["forecast_init_utc"].unique()[:5])

print("\n예측 관측소:")
print(sorted(predictions["station_id"].unique()))

# AWS 실제 관측자료 확인
print("\n===== AWS 실측자료 =====")
print("실제 관측시각:")
print(
    aws["valid_time_kst"]
    .drop_duplicates()
    .sort_values()
    .head(5)
    .tolist()
)

print("\n실제 관측소:")
print(sorted(aws["station_id"].unique()))

# 날짜 범위 확인
print("\n===== 날짜 범위 =====")

print(
    "예측 시작:",
    predictions["forecast_init_utc"].min()
)

print(
    "예측 종료:",
    predictions["forecast_init_utc"].max()
)

print(
    "실측 시작:",
    aws["valid_time_kst"].min()
)

print(
    "실측 종료:",
    aws["valid_time_kst"].max()
)

In [ ]:
# ==========================================
# 셀 9. AI 예측값과 AWS 실측값 연결
# ==========================================

import pandas as pd
import numpy as np

# 1. 데이터 복사
pred = predictions.copy()
obs = aws.copy()

# 2. AI 예측 시작시간 UTC 변환
pred["forecast_init_utc"] = pd.to_datetime(
    pred["forecast_init_utc"],
    utc=True,
    errors="raise"
)

# 3. 24시간 뒤 예측 유효시간 계산
# UTC 00:00 + 24시간 = 다음날 한국시간 09:00
pred["valid_time_kst"] = (
    pred["forecast_init_utc"]
    + pd.Timedelta(hours=int(LEAD_HOURS))
).dt.tz_convert("Asia/Seoul")

# 4. AWS 관측시간 정리
obs["valid_time_kst"] = pd.to_datetime(
    obs["valid_time_kst"],
    errors="raise"
)

# 이미 한국시간으로 변환되어 있다면 재변환하지 않음
if obs["valid_time_kst"].dt.tz is None:
    obs["valid_time_kst"] = (
        obs["valid_time_kst"]
        .dt.tz_localize("Asia/Seoul")
    )
else:
    obs["valid_time_kst"] = (
        obs["valid_time_kst"]
        .dt.tz_convert("Asia/Seoul")
    )

# 5. 관측소 번호 통일
pred["station_id"] = pd.to_numeric(
    pred["station_id"], errors="raise"
).astype(int)

obs["station_id"] = pd.to_numeric(
    obs["station_id"], errors="raise"
).astype(int)

# 6. 예측시간과 관측시간 검증
for name, frame in [
    ("AI 예측", pred),
    ("AWS 관측", obs)
]:
    valid_time = frame["valid_time_kst"]

    if not (
        (valid_time.dt.hour == 9)
        & (valid_time.dt.minute == 0)
    ).all():
        raise ValueError(
            f"{name}: 오전 9시가 아닌 자료가 포함되어 있습니다."
        )

# 7. 한국 날짜만 추출
pred["valid_date_kst"] = (
    pred["valid_time_kst"]
    .dt.strftime("%Y-%m-%d")
)

obs["valid_date_kst"] = (
    obs["valid_time_kst"]
    .dt.strftime("%Y-%m-%d")
)

# 8. 실제 공통 날짜와 관측소 확인
common_dates = (
    set(pred["valid_date_kst"])
    & set(obs["valid_date_kst"])
)

common_stations = (
    set(pred["station_id"])
    & set(obs["station_id"])
)

print("===== 데이터 연결 진단 =====")
print("예측 날짜:", pred["valid_date_kst"].nunique())
print("실측 날짜:", obs["valid_date_kst"].nunique())
print("공통 날짜:", len(common_dates))
print("공통 관측소:", len(common_stations))

print(
    "예측 날짜 예시:",
    sorted(pred["valid_date_kst"].unique())[:5]
)

print(
    "실측 날짜 예시:",
    sorted(obs["valid_date_kst"].unique())[:5]
)

# 공통 날짜 또는 관측소가 없으면 중단
if not common_dates:
    raise ValueError(
        "예측과 AWS 실측의 날짜가 겹치지 않습니다. "
        "평가연도와 AWS CSV 파일을 확인하세요."
    )

if not common_stations:
    raise ValueError(
        "예측과 AWS 실측의 관측소 번호가 겹치지 않습니다."
    )

# 9. AWS 중복 관측 검사
join_keys = ["station_id", "valid_date_kst"]

if obs.duplicated(join_keys).any():
    raise ValueError(
        "AWS에 동일 관측소·날짜의 자료가 중복되어 있습니다."
    )

# 10. AI 예측값과 AWS 관측값 결합
evaluation = pred.merge(
    obs[
        ["station_id", "valid_date_kst", "actual_c"]
    ],
    on=["station_id", "valid_date_kst"],
    how="left",
    validate="many_to_one"
)

print("\n===== 실측값 연결 현황 =====")

display(
    evaluation.groupby("model").agg(
        전체예측=("station_id", "size"),
        실측확보=("actual_c", "count")
    )
)

# 11. 세 모델 공통 데이터 생성
wide = evaluation.pivot(
    index=["station_id", "valid_time_kst"],
    columns="model",
    values="prediction_c"
)

actual = (
    evaluation.dropna(subset=["actual_c"])
    .drop_duplicates(
        ["station_id", "valid_time_kst"]
    )
    .set_index(
        ["station_id", "valid_time_kst"]
    )["actual_c"]
)

wide = wide.join(actual, how="inner")

required_models = list(MODELS.keys())

paired = wide.dropna(
    subset=required_models + ["actual_c"]
).reset_index()

if paired.empty:
    raise ValueError(
        "세 모델의 공통 평가 데이터가 없습니다. "
        "관측소별 날짜 일치 여부를 확인하세요."
    )

print("\n===== 최종 공통 평가자료 =====")
print("평가 건수:", len(paired))
print(
    "평가 날짜:",
    paired["valid_time_kst"].dt.date.nunique()
)
print("관측소:", paired["station_id"].nunique())

display(paired.head())

# 12. CSV 저장
paired.to_csv(
    TABLE_DIR / "common_evaluation_data.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n공통 평가자료 저장 완료!")

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score
)

def evaluate_regression(actual, predicted):

    y = np.asarray(actual, dtype=float)
    p = np.asarray(predicted, dtype=float)

    error = p - y
    abs_error = np.abs(error)

    return {
        "N": len(y),

        "MAE": mean_absolute_error(y, p),

        "RMSE": np.sqrt(
            mean_squared_error(y, p)
        ),

        "R2": (
            r2_score(y, p)
            if len(y) >= 2 and np.std(y) > 0
            else np.nan
        ),

        "Bias": np.mean(error),

        "MedAE": median_absolute_error(y, p),

        "Within_2C_pct": (
            np.mean(abs_error <= 2) * 100
        ),

        "P90_Abs_Error": np.percentile(
            abs_error, 90
        )
    }


rows = []

for model_name in MODELS:

    result = evaluate_regression(
        paired["actual_c"],
        paired[model_name]
    )

    rows.append({
        "Model": model_name,
        **result
    })

scores = pd.DataFrame(rows).sort_values(
    ["MAE", "RMSE"]
).reset_index(drop=True)

print("===== 3개 회귀모델 성능평가 =====")
display(scores.round(4))

scores.to_csv(
    TABLE_DIR / "regression_scores.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
season_map = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
}

paired["Season"] = (
    paired["valid_time_kst"]
    .dt.month.map(season_map)
)

season_rows = []

for season, group in paired.groupby("Season"):

    for model_name in MODELS:

        result = evaluate_regression(
            group["actual_c"],
            group[model_name]
        )

        season_rows.append({
            "Season": season,
            "Model": model_name,
            **result
        })

season_scores = pd.DataFrame(season_rows)

print("===== 계절별 성능 =====")
display(season_scores.round(3))

season_scores.to_csv(
    TABLE_DIR / "season_scores.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
station_rows = []

for station_id, group in paired.groupby("station_id"):

    for model_name in MODELS:

        result = evaluate_regression(
            group["actual_c"],
            group[model_name]
        )

        station_rows.append({
            "station_id": station_id,
            "Model": model_name,
            **result
        })

station_scores = pd.DataFrame(station_rows)

print("===== 관측소별 성능 =====")
display(station_scores.round(3))

station_scores.to_csv(
    TABLE_DIR / "station_scores.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# MAE / RMSE 비교
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(scores))
width = 0.35

bar1 = ax.bar(
    x - width / 2,
    scores["MAE"],
    width,
    label="MAE"
)

bar2 = ax.bar(
    x + width / 2,
    scores["RMSE"],
    width,
    label="RMSE"
)

ax.set_xticks(x)
ax.set_xticklabels(scores["Model"])

ax.set_ylabel("Error (°C)")
ax.set_title("Weather AI Regression Performance")
ax.legend()
ax.grid(axis="y", alpha=0.25)

ax.bar_label(bar1, fmt="%.3f")
ax.bar_label(bar2, fmt="%.3f")

plt.tight_layout()

plt.savefig(
    FIG_DIR / "mae_rmse_comparison.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()


# 실제 기온 vs 예측 기온
for model_name in MODELS:

    fig, ax = plt.subplots(figsize=(6, 6))

    ax.scatter(
        paired["actual_c"],
        paired[model_name],
        alpha=0.25,
        s=12
    )

    lower = min(
        paired["actual_c"].min(),
        paired[model_name].min()
    )

    upper = max(
        paired["actual_c"].max(),
        paired[model_name].max()
    )

    ax.plot(
        [lower, upper],
        [lower, upper],
        linestyle="--"
    )

    ax.set_xlabel("Observed Temperature (°C)")
    ax.set_ylabel("Predicted Temperature (°C)")
    ax.set_title(model_name)

    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.2)

    plt.tight_layout()

    filename = model_name.replace("-", "_")

    plt.savefig(
        FIG_DIR / f"{filename}_scatter.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()